<a href="https://colab.research.google.com/github/pedromp-hub/Practica-An-lisis-De-Sentimientos/blob/main/Pr%C3%A1ctica_2_1_sentimientos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Qué es el análisis de sentimiento**

Es una tarea de clasificación de texto. Dado un texto, el modelo decide si la intención utilizada es positiva, negativa o neutral.

No es "**entender**" el texto como lo haría una persona. Es detectar patrones de palabras y contexto que, estadísticamente, se asocian a cada categoría.

Para esta práctica en español, el modelo más adecuado es **RoBERTuito**. Es un modelo entrenado específicamente con **500 millones de tweets en español**, lo que le permite entender jerga, emojis, abreviaturas y texto informal.

Está diseñado para redes sociales, que es donde el análisis de sentimiento tiene más aplicación real. Y funciona bien en tareas de sentimiento, odio e ironía en español.

RoBERTuito no se entrenó desde cero. Es un modelo preentrenado: alguien invirtió muchísimo tiempo y recursos en entrenarlo con textos masivos para que "entienda" el español.

Luego ese modelo base se ajustó (mediante fine-tuning) con un conjunto más pequeño de tweets etiquetados como positivos, negativos o neutros. El resultado es un modelo especializado en análisis de sentimiento en español, listo para usar.

Esto es importante porque explica por qué tú no entrenas nada. Usas un modelo que ya existe, ya está entrenado, y solo lo llamas desde Colab. Es exactamente lo que haría un desarrollador profesional: **no reinventar la rueda, usar lo que ya funciona**.

# **Hugging Face**

Hugging Face es la plataforma donde viven estos modelos. Su función **pipeline** es una abstracción: tú le dices "quiero realizar un análisis de sentimiento" y "usa este modelo", y ella se encarga de todo lo demás (preprocesar el texto, pasarlo por el modelo, devolver el resultado).

Solo se necesita entender que pipeline es el "botón" que conecta el texto con un modelo ya entrenado.

El modelo devuelve dos cosas:

    Una etiqueta: POS, NEG o NEU

    Una confianza (score), un número entre 0 y 1

La confianza no es lo mismo que **"acierto"**. Es la seguridad del modelo en su propia predicción. Un score de 0.99 no garantiza que tenga razón; un score de 0.51 en una frase ambigua es normal y no es un fallo.

El modelo no lee intenciones, lee patrones. El guiño o el signo de interrogación alteran el contexto que el modelo aprendió durante el preentrenamiento, y eso mueve la predicción.

In [1]:
from transformers import pipeline

clasificador = pipeline("sentiment-analysis", model="pysentimiento/robertuito-sentiment-analysis")
# carga un modelo de inteligencia artificial
# entrenado para analizar el sentimiento
# de textos en español, con especial enfoque
# en lenguaje informal y redes sociales

# === este es el texto  que se clasifica ===
texto = "Me encanta este módulo de IA"
# =====================================

resultado = clasificador(texto)
print(resultado)

config.json:   0%|          | 0.00/925 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  435MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

[{'label': 'POS', 'score': 0.9514580368995667}]


In [2]:
# clasifica un texto que sea negativo

texto = "que aburrido estoy"
resultado = clasificador(texto)
print(resultado)

[{'label': 'NEG', 'score': 0.9639661312103271}]


In [3]:
# clasifica un texto neutro
texto = "pan"
resultado = clasificador(texto)
print(resultado)

[{'label': 'NEU', 'score': 0.5387777090072632}]


In [4]:
# clasifica un texto irónico
texto = "tienes muchas luces"
resultado = clasificador(texto)
print(resultado)

[{'label': 'POS', 'score': 0.5579562187194824}]


In [5]:
# clasifica un texto usando emojis
texto = "🙈"
resultado = clasificador(texto)
print(resultado)

[{'label': 'NEU', 'score': 0.4858541488647461}]


**¿Cómo funciona el análisis de sentimientos? ¿Qué límites tiene?**

El análisis de sentimientos es como una inteligencia artificial que lee lo que escribes y deduce si estás contento, molesto o neutral. Lo hace buscando ciertas palabras clave para calcular qué emoción transmite el mensaje. El problema es que se le da muy mal entender las bromas, el sarcasmo o el doble sentido. Tampoco entiende el contexto, por lo que puede confundirse fácilmente. Debido a esto, muchas veces termina bloqueando mensajes normales por error.

# **Práctica**

Implementa en python un analizador de comentarios. Los comentarios positivos y nuetros serán aceptados, salvo que contengan alguna palabra de la lista negra de palabras. Los comentarios negativos que contengan alguna palabra de la lista negra o que tengan un score de más de 0.5 serán rechazadas.

In [7]:
# pon aquí el código de tu analizador

from transformers import pipeline
import re

# 1. Cargar el modelo de análisis de sentimientos
clasificador = pipeline("sentiment-analysis", model="pysentimiento/robertuito-sentiment-analysis")

# 2. Definir la lista negra de palabras no admitidas
lista_negra = ["mierda", "pendejo", "tonto", "cabron", "subnormal", "enano", "idiota", "puta", "maricon", "puton"]

def analizar_comentario(comentario):
    comentario_lower = comentario.lower()

    # Comprobar si contiene alguna palabra de la lista negra
    tiene_palabra_prohibida = False
    palabra_encontrada = ""
    for palabra in lista_negra:
        if re.search(rf"\b{palabra}\b", comentario_lower):
            tiene_palabra_prohibida = True
            palabra_encontrada = palabra
            break

    # Obtener el análisis de sentimiento del modelo
    resultado = clasificador(comentario)
    label = resultado[0]['label']   # 'POS', 'NEG' o 'NEU'
    score = resultado[0]['score']   # Confianza del modelo

    # REGLA 1: Negativos que tengan palabra prohibida o score > 0.5 se rechazan
    if label == 'NEG' and (tiene_palabra_prohibida or score > 0.5):
        motivo = f"Rechazado por ser negativo (score: {score:.2f})"
        if tiene_palabra_prohibida:
            motivo += f" y contener la palabra prohibida '{palabra_encontrada}'"
        return "RECHAZADO", motivo

    # REGLA 2: Si contiene palabra prohibida (independientemente de si el modelo lo vio positivo/neutro)
    if tiene_palabra_prohibida:
        return "RECHAZADO", f"Rechazado por contener la palabra prohibida '{palabra_encontrada}'"

    # REGLA 3: Positivos y neutros limpios son aceptados
    return "ACEPTADO", f"Aceptado (Sentimiento: {label}, score: {score:.2f})"

# --- PRUEBAS ---
comentarios_de_prueba = [
    "Me encantó el servicio, muchas gracias.", # Positivo limpio -> Aceptado
    "El producto es aceptable, cumple su función.", # Neutro limpio -> Aceptado
    "Esto es una completa mierda.", # Negativo con palabra prohibida -> Rechazado
    "No me ha gustado nada, me parece horrible.", # Negativo con score alto (> 0.5) -> Rechazado
    "El servicio estuvo genial pero la comida me pareció una mierda.",
    "Llegó en el plazo previsto y el empaque está bien.",
    "Excelente atención por parte del personal, super atento.",
    "Pésimo producto, me siento muy decepcionado y frustrado.",
    "No atiendes bien, eres un tonto.",
    "El paquete llegó muy rápido y en perfecto estado, gracias."


]

for c in comentarios_de_prueba:
    estado, detalle = analizar_comentario(c)
    print(f"Comentario: \"{c}\"")
    print(f"Resultado: {estado} -> {detalle}\n")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Comentario: "Me encantó el servicio, muchas gracias."
Resultado: ACEPTADO -> Aceptado (Sentimiento: POS, score: 0.98)

Comentario: "El producto es aceptable, cumple su función."
Resultado: ACEPTADO -> Aceptado (Sentimiento: POS, score: 0.78)

Comentario: "Esto es una completa mierda."
Resultado: RECHAZADO -> Rechazado por ser negativo (score: 0.97) y contener la palabra prohibida 'mierda'

Comentario: "No me ha gustado nada, me parece horrible."
Resultado: RECHAZADO -> Rechazado por ser negativo (score: 0.98)

Comentario: "El servicio estuvo genial pero la comida me pareció una mierda."
Resultado: RECHAZADO -> Rechazado por ser negativo (score: 0.90) y contener la palabra prohibida 'mierda'

Comentario: "Llegó en el plazo previsto y el empaque está bien."
Resultado: ACEPTADO -> Aceptado (Sentimiento: POS, score: 0.84)

Comentario: "Excelente atención por parte del personal, super atento."
Resultado: ACEPTADO -> Aceptado (Sentimiento: POS, score: 0.96)

Comentario: "Pésimo producto, me 